In [ ]:
from google.colab import drive

from keras.optimizers import Adam, SGD
from keras.callbacks import TensorBoard, TerminateOnNaN, EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from keras import layers
from keras.layers import (
    Input, Add, Dense, Activation, ZeroPadding2D, BatchNormalization,
    Flatten, Conv2D, AveragePooling2D, MaxPooling2D, ReLU, Dropout,
    GlobalMaxPooling2D, GlobalAveragePooling2D
)
from keras.models import Model
from keras.initializers import glorot_uniform
import tensorflow as tf
import numpy as np
import scipy.io as sp
from scipy.ndimage import shift
import matplotlib.pyplot as plt
from matplotlib import image
from PIL import Image, ImageOps
import os
import io
import gc
import random
import zipfile
import pandas as pd
from sklearn.model_selection import train_test_split

In [ ]:
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


AlexNet

In [ ]:
class AlexNet:
    def AlexNet(input_shape, outputs, activation):
        # Input Layer
        X_input = Input(input_shape, name = "AlexNet Input")

        # Layer 1 - Convolutions
        X = Conv2D(filters=64, kernel_size=3, strides=1, padding="same")(X_input)
        X = Activation('relu')(X)

        # Layer 2 - Convolutions
        X = Conv2D(filters=64, kernel_size=3, strides=1, padding="same")(X)
        X = Activation('relu')(X)
        X = MaxPooling2D(pool_size=3, strides=2)(X)

        # Layer 3 - Convolutions
        X = Conv2D(filters=32, kernel_size=3, strides=1, padding="valid")(X)
        X = Activation('relu')(X)
        X = MaxPooling2D(pool_size=3, strides=2)(X)

        # Layer 4 - Convolutions
        X = Conv2D(filters=32, kernel_size=3, strides=1, padding="valid")(X)
        X = Activation('relu')(X)
        X = MaxPooling2D(pool_size=3, strides=2)(X)

        # Layer 5 - Convolutions
        X = Conv2D(filters=32, kernel_size=3, strides=1, padding="valid")(X)
        X = Activation('relu')(X)
        X = MaxPooling2D(pool_size=3, strides=2)(X)


        # Layer 6 - Dense
        X = Flatten()(X)
        X = Dense(units=64)(X)
        X = Activation('relu')(X)

        # Layer 7 - Dense
        X = Dense(units=32)(X)
        X = Activation('relu')(X)

        # Layer 8 - Dense
        X = Dense(outputs, activation=activation, name='fc' + str(outputs), kernel_initializer = glorot_uniform(seed=0))(X)


        # Create model
        model = Model(inputs = X_input, outputs = X, name='AlexNet_model')

        return model

Image Processing

In [ ]:
def introduce_gaps(coords, gap_size=500):
    '''
    This function introduces gaps of 500 consecutive points.
    '''
    segments = []
    i = 0
    n = len(coords)
    k = np.random.randint(0, 4)
    option = [10000, 5000, 2500, 1500]
    consecutive = option[k]

    while i < n:
        end = min(i + consecutive, n)
        segments.append(coords[i:end])
        i = end + gap_size
    return segments

matplotlib.use("Agg")

def rasterize_segments(segments, img_size=(515, 389)):
    '''
    Given the coordinates, it creates a binary image of specific dimensions.
    '''
    fig, ax = plt.subplots(figsize=(img_size[0]/100, img_size[1]/100), dpi=100)
    ax.set_axis_off()
    ax.set_aspect('equal', adjustable='box')

    # Calculate global bounding box for all segments to normalize scaling
    all_coords = np.vstack(segments)
    min_x, max_x = all_coords[:, 0].min(), all_coords[:, 0].max()
    min_y, max_y = all_coords[:, 1].min(), all_coords[:, 1].max()

    # Adjust plot limits to fit all segments tightly
    ax.set_xlim(min_x, max_x)
    ax.set_ylim(min_y, max_y)

    for seg in segments:
        if len(seg) > 1:
            ax.plot(seg[:,0], seg[:,1], color='black', linewidth=1)

    plt.subplots_adjust(left=0, right=1, top=1, bottom=0)

    # Render the figure to a buffer
    fig.canvas.draw()
    img_rgba = np.array(fig.canvas.renderer.buffer_rgba())
    plt.close(fig)

    # I delete the figure to avoid memory saturation
    del fig, ax
    if np.random.randint(0, 100) == 0:
      gc.collect()

    # Convert RGBA to grayscale
    img = Image.fromarray(img_rgba).convert("L")
    img = img.resize(img_size, Image.LANCZOS)

    # Binarize the image
    img = img.point(lambda p: 255 if p > 128 else 0)

    return np.array(img)

Data Augmentation

In [ ]:
def add_pepper(img_bin: np.ndarray, amount: float, seed: int | None = None) -> np.ndarray:
    """
    Add black pixel (pepper) to a binary image.
    img_bin: binary array {0,1} o {0,255}, 2D.
    amount: fraction of pixels to convert (0 -> 1.0).
    """
    noisy = img_bin.copy()
    h, w = noisy.shape[:2]
    total = h * w
    k = int(round(amount * total))
    if k == 0:
        return noisy

    rng = np.random.default_rng(seed)
    idx = rng.choice(total, size=k, replace=False)

    black = 0
    noisy.flat[idx] = black
    return noisy

In [ ]:
def translate_image(img, tx, ty, mode='constant', cval=1):
    """
    Traslate a pixel image.
    img: array (H,W) or (H,W,1)
    tx, ty: how much it translate in pixel (columns, rows)
    cval=1 => white background after translation
    """
    if img.ndim == 3:
        return shift(img, shift=(ty, tx, 0), mode=mode, cval=cval)
    else:
        return shift(img, shift=(ty, tx), mode=mode, cval=cval)

Custom generator for traing and validation

In [ ]:
def custom_generator(dataframe, batch_size, Apply_DataAug):
    '''
    I create a custom generator which extract a batch of randomly chosen images.
    For each of them it applies data augmentation
    '''

    while True:
        batch = {'images': [], 'label': []}
        dataframe = dataframe.sample(frac=1).reset_index(drop=True)

        for b in range(batch_size):

      # Get the "image" from the dataframe and store it as the original array
            fname = dataframe["Percorso_Coordinate"][b]
            coords = np.load(os.path.join(fname))

            target_width = 515
            target_height = 389

            # random choice between taking a segment or inserting gaps in the curve
            value = np.random.randint(0, 2)

            if value == 0:
                portion = np.random.randint(1000, 20000)
                start = np.random.randint(0, 20001-porzione)
                end = start + portion
                coordss = coords[start:end]
                img = rasterize_segments([coordss], img_size=(target_width, target_height))

            else:
                segments = introduce_gaps(coords, gap_size=500)
                img = rasterize_segments(segments, img_size=(target_width, target_height))

            threshold = 200

            pil_img = Image.fromarray(img)

            # Rotation

            rotation_angle = np.random.uniform(0, 359)
            width, height = pil_img.size
            diagonal = int(np.sqrt(width**2 + height**2))
            padded_image = ImageOps.expand(pil_img, border=(diagonal - width) // 2, fill=255)
            rotated_image = padded_image.rotate(
                rotation_angle,
                resample=Image.NEAREST,
                fillcolor=255
            )

            # Central Crop
            width, height = rotated_image.size
            left = (width - target_width) // 2
            top = (height - target_height) // 2
            right = left + target_width
            bottom = top + target_height
            rotated_image = rotated_image.crop((left, top, right, bottom))

            # Convert it to a binary array for safety
            rotated_array = np.array(rotated_image)
            rotated_array = (rotated_array > threshold).astype(np.uint8) * 255


            # Traslation
            tx = np.random.randint(-25, 25)
            ty = np.random.randint(-25, 25)
            img_trans = translate_image(np.array(rotated_array), tx, ty, cval=255)

            # Pepper noise
            tipo = np.random.uniform(0.0, 0.075)
            img_pepper = add_pepper(img_trans, amount=tipo)

            img_pepper = (img_pepper > 200).astype(np.float32)

            image = img_pepper[:, :, np.newaxis]


            label = dataframe["Dimensione_Frattale"][b]
            batch['images'].append(image)
            batch['label'].append([label])


        batch['images'] = np.asarray(batch['images'])
        batch['label'] = np.array(batch['label'])

        yield batch['images'], batch['label']

Validation

In [ ]:
def _py_func(path, label):
    path = path.decode("utf-8")
    coords = np.load(path)

    value = np.random.randint(0, 2)
    if value == 0:

        portion = np.random.randint(1000, 20000)
        start = np.random.randint(0, 20001-porzione)
        end = start + portion
        coordss = coords[start:end]
        img = rasterize_segments([coordss], img_size=(515, 389))
    else:
        segments = introduce_gaps(coords, gap_size=500)
        img = rasterize_segments(segments, img_size=(515, 389))

    tipo = np.random.uniform(0.0, 0.075)
    img_pepper = add_pepper(img, amount=tipo)
    img_pepper = (img_pepper > 200).astype(np.float32)

    image = img_pepper[:, :, np.newaxis]

    return image.astype(np.float32), np.float32(label)

def preprocess_val(path, label):

    image, label = tf.numpy_function(
        func=_py_func,
        inp=[path, label],
        Tout=[tf.float32, tf.float32]
    )

    image.set_shape((389, 515, 1))
    label.set_shape(())

    return image, label

Uploading the Dataset

In [ ]:
drive_base = "/content/drive/My Drive/coord"
local_base = "/content/dataset"
os.makedirs(local_base, exist_ok=True)

def upload_modify_csv(path_csv, fil):
    df = pd.read_csv(path_csv)
    df["Percorso_Coordinate"] = df["Percorso_Coordinate"].str.replace(
        f"/content/drive/My Drive/{fil}/",
        f"/content/dataset/{fil}/{fil}/"
    )
    return df

# List of files
folder = [f"DATASET-{i}" for i in range(1,8)]


# Extract the ZIP files locally
for fil in folder:
    zip_path = os.path.join(drive_base, f"{fil}.zip")
    estrai_in = os.path.join(local_base, fil)
    if os.path.exists(zip_path) and not os.path.exists(estrai_in):
        os.makedirs(estrai_in, exist_ok=True)
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(estrai_in)
        print(f"Extract: {fil}.zip")

# Uploading and modifing the original CSV

df_gruops = []
for idx in range(1,8):
    group = [
        f"DATASET-{idx}"
    ]
    df_list = []
    for fil in group:
        path_csv = os.path.join(local_base, fil, fil, "a_etichette.csv")
        if os.path.exists(path_csv):
            df_modificato = upload_modify_csv(path_csv, fil)
            df_list.append(df_modificato)
    if df_list:
     df_group = pd.concat(df_list, ignore_index=True)
     df_gruops.append(df_group)
    else:
     print(f"No CSV found for this group {idx}")

# Putting everything togheter

df = pd.concat(df_gruops, ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)

# Dividing the dataset in local training + (validation + test) CSV files

train_size = 0.71429
val_size = 0.142855
test_size = 0.142855

train_dff, df_temp = train_test_split(df, test_size=(1 - train_size), random_state=300)

val_dff, test_dff = train_test_split(df_temp, test_size=(test_size / (val_size + test_size)), random_state=300)

train_dff.to_csv("/content/train_df_local.csv", index=False)

val_dff.to_csv("/content/val_df_local.csv", index=False)

test_dff.to_csv("/content/test_df_local.csv", index=False)

#  Final Sets

train_df = pd.read_csv("/content/train_df_local.csv")
val_df = pd.read_csv("/content/val_df_local.csv")
test_df = pd.read_csv("/content/test_df_local.csv")

Mounted at /content/drive
✅ Estratto: DATASET-1.zip
✅ Estratto: DATASET-2.zip
✅ Estratto: DATASET-3.zip
✅ Estratto: DATASET-4.zip
✅ Estratto: DATASET-5.zip
✅ Estratto: DATASET-6.zip
✅ Estratto: DATASET-7.zip


In [ ]:
print(f"Training set size: {len(train_df)}")
print(f"Validation set size: {len(val_df)}")
print(f"Test set size: {len(test_df)}")

Training set size: 15000
Validation set size: 3000
Test set size: 3000


Visualization of Dataset Distribution

In [ ]:
data1 = train_df["Dimensione_Frattale"]
data2 = val_df["Dimensione_Frattale"]
data3 = test_df["Dimensione_Frattale"]

num_bins = 100

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Training
axes[0].hist(data1, bins=num_bins, color='blue', alpha=0.7)
axes[0].set_title('Training')
axes[0].set_xlabel('Values')
axes[0].set_ylabel('Frequence')

# Validation
axes[1].hist(data2, bins=num_bins, color='green', alpha=0.7)
axes[1].set_title('Validation')
axes[1].set_xlabel('Values')
axes[1].set_ylabel('Frequence')

# Test
axes[2].hist(data3, bins=num_bins, color='orange', alpha=0.7)
axes[2].set_title('Test')
axes[2].set_xlabel('Values')
axes[2].set_ylabel('Frequence')


plt.tight_layout()

plt.show()

Uploading training and validation

In [ ]:
batch_size = 32


train_set= custom_generator(train_df, batch_size, Apply_DataAug = True)


image_paths_val = val_df["Percorso_Coordinate"].astype(str).to_numpy()
labels_val = val_df["Dimensione_Frattale"].astype("float32").to_numpy()

valid_set = tf.data.Dataset.from_tensor_slices((image_paths_val, labels_val))
valid_set = valid_set.map(lambda path, label: preprocess_val(path, label), num_parallel_calls=tf.data.AUTOTUNE, deterministic=True)
valid_set = valid_set.batch(batch_size, drop_remainder=False)
valid_set = valid_set.prefetch(tf.data.AUTOTUNE)

train_steps = len(train_df) // batch_size
valid_steps = len(val_df) // batch_size

Compiling

In [ ]:
path = '/content/drive/My Drive/Modelli/NuoveDimensionii/' # path where you wanna save your model parameters and training history
file_name = 'AlexNet'

optimizer = Adam(learning_rate=1e-3, beta_1=0.9, beta_2=0.999, epsilon=1e-8, amsgrad=False)

model = AlexNet.AlexNet(input_shape=(389, 515, 1),outputs=1, activation='linear')
model.compile(optimizer=optimizer, loss='mse', metrics=['mse'])

model_checkpoint = ModelCheckpoint(path+file_name+'_model.keras', monitor='val_mse', verbose=1, save_best_only=True)
reduce_lr = ReduceLROnPlateau('val_mse', factor=0.999, patience=int(4), cooldown=0, min_lr=1e-6, verbose=1)
early_stop = EarlyStopping('val_mse', patience=20, verbose=1)


terminate = TerminateOnNaN()
callbacks = [model_checkpoint, early_stop, terminate]

print(f"Number of parameters: {model.count_params():,}".replace(",", "."))

Totale parametri: 1.323.937


Model Running

In [ ]:
epochs = 60  #  <--------------------------------- occhio
history = model.fit(train_set,
                    epochs = epochs,
                    batch_size = batch_size,
                    steps_per_epoch = train_steps,
                    validation_data= valid_set,
                    validation_steps = valid_steps,
                    callbacks=callbacks)

Epoch 1/60
  5/468 ━━━━━━━━━━━━━━━━━━━━ 7:04:08 55s/step - loss: 1.4789 - mse: 1.4789

Save History

In [ ]:
np.save(path+file_name,history.history)